In [37]:
# Importar bibliotecas

import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score

In [38]:
# Carregar os dados
df_costs = pd.read_csv('datasets/healthcosts_cleaned.csv')

In [39]:
# Mostrar as primeiras linhas
df_costs.head(10)

,age,sex,bmi,children,smoker,region,medical charges
0,19,female,27.900,0,1,southwest,16884.92400
1,18,male,33.770,1,0,southeast,1725.55230
2,28,male,33.000,3,0,southeast,4449.46200
3,33,male,22.705,0,0,northwest,21984.47061
4,32,male,28.880,0,0,northwest,3866.85520
5,31,female,25.740,0,0,southeast,3756.62160
6,46,female,33.440,1,0,southeast,8240.58960
7,37,female,27.740,3,0,northwest,7281.50560
8,37,male,29.830,2,0,northeast,6406.41070
9,60,female,25.840,0,0,northwest,28923.13692


In [40]:
# Mostrar as últimas linhas
df_costs.tail(10)

,age,sex,bmi,children,smoker,region,medical charges
1328,23,female,24.225,2,0,northeast,22395.74424
1329,52,male,38.600,2,0,southwest,10325.20600
1330,57,female,25.740,2,0,southeast,12629.16560
1331,23,female,33.400,0,0,southwest,10795.93733
1332,52,female,44.700,3,0,southwest,11411.68500
1333,50,male,30.970,3,0,northwest,10600.54830
1334,18,female,31.920,0,0,northeast,2205.98080
1335,18,female,36.850,0,0,southeast,1629.83350
1336,21,female,25.800,0,0,southwest,2007.94500
1337,61,female,29.070,0,1,northwest,29141.36030


In [41]:
# Mostrar a estrutura
df_costs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   age              1338 non-null   int64  
 1   sex              1338 non-null   object 
 2   bmi              1338 non-null   float64
 3   children         1338 non-null   int64  
 4   smoker           1338 non-null   int64  
 5   region           1338 non-null   object 
 6   medical charges  1338 non-null   float64
dtypes: float64(2), int64(3), object(2)
memory usage: 73.3+ KB


# Preparação dos dados

In [42]:
# Preparar os dados para o modelo
X = df_costs.drop(columns=['medical charges'])
y = df_costs['medical charges']

In [43]:
# Importar preprocessor já salvo anteriormente
import joblib
preprocessor = joblib.load('preprocessor_dataset_healthcosts.pkl')

In [44]:
# Dividir os dados em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=51)

In [45]:
# Aplicar preprocessor nos dados de treinamento e teste
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [46]:
# Mostrar as dimensões dos conjuntos
print(f'Dados de Treinamento: {X_train.shape}')
print(f'Dados de Teste: {X_test.shape}')

Dados de Treinamento: (1070, 10)
Dados de Teste: (268, 10)


# Treinamento do Modelo

In [77]:
# Criar o modelo de AdaBoost Regressor
boosting_model = AdaBoostRegressor(
    estimator=LinearRegression(),
    n_estimators=50,
    learning_rate=0.1,
    random_state=51
)

In [78]:
# Treinar o modelo
boosting_model.fit(X_train, y_train)

AdaBoostRegressor(estimator=LinearRegression(), learning_rate=0.1,
                  random_state=51)

# Análise dos Resultados

In [79]:
# Fazer predições no conjunto de testes
y_pred = boosting_model.predict(X_test)

In [80]:
# Mostrar y_pred
y_pred

array([10976.        , 37612.        ,  5012.93521639, 12765.00076501,
       34555.49924998, 12999.74799563, 13156.52947181, 16901.55092628,
        7744.        , 12640.        , 11358.1459602 , 13569.59966511,
       11520.        ,  6253.18413887,  6846.15958396, 14154.18198419,
        7670.43161232,  7217.98912424, 26525.82136294, 29036.30264288,
       12000.23933284, 10528.        , 33230.        , 14938.16413176,
        7447.92737838, 17696.        , 11513.71701492,  4768.        ,
       23471.60633861, 10052.57650688,  5920.        , 30836.93665273,
        7639.24864574,  6403.13043574,  9638.78991361, 12923.74178716,
       15333.53132246,  4336.13914346, 14229.58252536,  9851.23072431,
       11755.79385553,  2417.94299778,  7474.32034804,  4121.64499419,
        6047.80488374, 16718.22352656, 16912.81171564, 35948.17542038,
        9313.51307181, 14148.75500735,  7376.81323911, 31345.82909   ,
        9073.21359529, 40906.21777261,  5476.34472979, 28384.        ,
      

In [81]:
# Avaliar métricas do modelo
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [82]:
# Mostrar as métricas
print(f'Root Mean Squared Error: {rmse}')
print(f'R2: {r2}')

Root Mean Squared Error: 6776.479230900867
R2: 0.7359098647707016


## Análise de importancia das features

In [83]:
# Calcular a importância das features usando os coeficientes

# Obter os coeficientes de cada estimador
coefs = np.array([estimator.coef_ for estimator in boosting_model.estimators_])

In [84]:
# Calcular média dos coeficientes absolutos
importances = np.mean(np.abs(coefs), axis=0)

In [85]:
# Normalizar as importâncias
importances = importances / np.sum(importances)

In [86]:
# Obter os nomes das features
feature_names = preprocessor.get_feature_names_out()

In [87]:
# Criar um DataFrame com as importâncias e os nomes das features
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})

In [88]:
# Ordenar o DataFrame pela importância
importance_df = importance_df.sort_values('importance', ascending=True)

In [89]:
# Criar o gráfico de barras com a importância das features
fig = px.bar(importance_df,
             x='importance',
             y='feature',
             title='Importância das Features',
             orientation='h')

# fig.update_xaxes(tickangle=45)
fig.show()

# Propriedades do Modelo

In [90]:
# Quais foram os erros dos estimadores
boosting_model.estimator_errors_

array([0.13261321, 0.14178828, 0.13945046, 0.15740933, 0.15149962,
       0.1644491 , 0.16474001, 0.1676325 , 0.17836876, 0.18881481,
       0.19048031, 0.19066031, 0.19322675, 0.20531693, 0.21697667,
       0.21022059, 0.21494196, 0.23164484, 0.2310104 , 0.24199716,
       0.24903985, 0.25084348, 0.2591333 , 0.26771042, 0.30488533,
       0.28626092, 0.29762589, 0.28687927, 0.29702499, 0.30133027,
       0.32543678, 0.31196894, 0.32697554, 0.31564419, 0.37087937,
       0.32963035, 0.36215705, 0.34439344, 0.34570468, 0.35023888,
       0.39527036, 0.35869247, 0.37149859, 0.37045863, 0.42759561,
       0.38281458, 0.40230907, 0.3984117 , 0.36942582, 0.40315813])

In [91]:
# Pesos dos estimadores
boosting_model.estimator_weights_

array([0.18780483, 0.18005159, 0.18198617, 0.16776316, 0.17228874,
       0.16254902, 0.16233745, 0.16024999, 0.15274386, 0.14577297,
       0.14468923, 0.14457254, 0.14291783, 0.13533886, 0.12833726,
       0.13235963, 0.12953896, 0.11990467, 0.12026147, 0.11417611,
       0.11037397, 0.10941188, 0.10504781, 0.10062701, 0.08241411,
       0.09136138, 0.08586289, 0.09105893, 0.0861505 , 0.08409712,
       0.07288972, 0.07909304, 0.07218963, 0.07738624, 0.05284461,
       0.07098574, 0.05660141, 0.06437761, 0.06379739, 0.06179894,
       0.04252115, 0.05810437, 0.05257932, 0.05302498, 0.02916678,
       0.04776187, 0.03958532, 0.04120874, 0.05346808, 0.03923233])